# Procesamiento y Curación - Fantasy & Paranormal

Este notebook ejecuta la curación reproducible desde los JSON crudos hasta Parquet listo para modelado. La lógica pesada vive en `src/process_goodreads.py` y `src/utils/`.

## Decisiones de limpieza

- Strings vacíos se convierten a nulos antes de castear tipos.
- `rating = 0` en interacciones se trata como rating ausente, no como calificación negativa.
- Fechas Goodreads se parsean con zona horaria UTC y valores inválidos quedan nulos.
- El texto de reseña disponible es `review_text_incomplete`; se normaliza HTML y whitespace sin inventar contenido faltante.
- Los duplicados exactos se eliminan y, para `review_id`, se conserva la versión con `date_updated` más reciente.
- Los outliers no se eliminan; se agregan columnas capadas al p99 para modelado manteniendo las originales.

In [ ]:
from pathlib import Path
import os, sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.process_goodreads import process_category
from src.config import CATEGORIES

cfg = CATEGORIES['fantasy_paranormal']

In [ ]:
report = process_category('fantasy_paranormal', chunksize=100_000, bucket_count=256)
report

## Validación de salidas

Se comprueba que los Parquet sean legibles y que las invariantes principales se mantengan.

In [ ]:
books = pd.read_parquet(cfg.processed_dir / 'books_curated.parquet')
interactions = pd.read_parquet(cfg.processed_dir / 'interactions_curated.parquet')

assert books['book_id'].notna().all()
assert interactions['book_id'].notna().all()
assert interactions['rating_clean'].dropna().between(1, 5).all()
assert books['book_id'].duplicated().sum() == 0
assert interactions['review_id'].dropna().duplicated().sum() == 0

books.head(), interactions.head()